In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import h5py

VISUALIZATION_SAMPLE_SIZE = 2000
START_ROW = 0 # Starting Index
NUM_ROWS = 3 # how many rows to read

NB_ID = "01"

In [ ]:
def load_h5_batch(file_path, start_row=0, num_rows=1, num_samples=2000):
    """
    Loads a specific segment (row) from the MIT H5 file.
    Structure is (100, 43560) -> 100 segments of 43.56ms each.
    """
    with h5py.File(file_path, 'r') as f:
        # Access the (100, 43560) matrix
        dset = f['dataset'] 

        batch_data = dset[start_row : start_row + num_rows, :num_samples]
        
        if batch_data.ndim == 1:
            batch_data = batch_data.reshape(1, -1)
                
        return batch_data

In [ ]:
print(f"--- Loading Batch (Rows {START_ROW} to {START_ROW + NUM_ROWS - 1}) ---")

# Load Target Batch
print(f"Loading Target: {MIT_CLEAN_FILE.name}")
clean_batch = load_h5_batch(MIT_CLEAN_FILE, start_row=START_ROW, num_rows=NUM_ROWS, num_samples=VISUALIZATION_SAMPLE_SIZE)

# Load Noise Batch
print(f"Loading Noise: {MIT_NOISE_FILE.name}")
noise_batch = load_h5_batch(MIT_NOISE_FILE, start_row=START_ROW, num_rows=NUM_ROWS, num_samples=VISUALIZATION_SAMPLE_SIZE)

if clean_batch is not None:
    print(f"Clean Batch Loaded. Shape: {clean_batch.shape}, Type: {clean_batch.dtype}")
else:
    print("Failed to load Clean Batch")

In [ ]:
if clean_batch is not None and noise_batch is not None:
    # Create a grid of subplots: Rows = NUM_ROWS, Cols = 2
    fig, axes = plt.subplots(NUM_ROWS, 2, figsize=(15, 3.5 * NUM_ROWS), sharex=True)
    
    # Handle case where NUM_ROWS=1 (axes is 1D array)
    if NUM_ROWS == 1:
        axes = axes.reshape(1, -1)
        
    for i in range(NUM_ROWS):
        current_row_idx = START_ROW + i
        
        # --- Column 1: Clean Signal ---
        ax_clean = axes[i, 0]
        sig_data = clean_batch[i]
        ax_clean.plot(sig_data.real, color='C0', label='Real (I)', alpha=0.9)
        ax_clean.plot(sig_data.imag, color='C0', linestyle='--', label='Imag (Q)', alpha=0.4)
        ax_clean.set_title(f"Row {current_row_idx}: CommSignal2")
        ax_clean.set_ylabel("Amplitude")
        if i == 0: ax_clean.legend(loc="upper right")
        
        # --- Column 2: Noise Signal ---
        ax_noise = axes[i, 1]
        noise_data = noise_batch[i]
        ax_noise.plot(noise_data.real, color='C1', label='Real (I)', alpha=0.9)
        if np.iscomplexobj(noise_data):
            ax_noise.plot(noise_data.imag, color='C1', linestyle=':', label='Imag (Q)', alpha=0.4)
            
        ax_noise.set_title(f"Row {current_row_idx}: EMISignal1")
        if i == 0: ax_noise.legend(loc="upper right")
        
        # X-Axis Label (Bottom only)
        if i == NUM_ROWS - 1:
            ax_clean.set_xlabel("Sample Index")
            ax_noise.set_xlabel("Sample Index")

    plt.tight_layout()
    save_plot(f"mit_dataset_rows_{START_ROW}_to_{START_ROW + NUM_ROWS - 1}.png", machine_id="B", nb_id=NB_ID, fig_id="01")
    plt.show()
else:
    print("Cannot plot: Data missing.")

# Fig 00.1 Raw Components: The "Ingredients" of the Mixture

This visualization (Fig 1) isolates the raw input signals before mixing, revealing why the MIT dataset is an ideal training ground for deep learning-based jamming mitigation.

### The Equalizer: Unity Power Normalization
The most critical observation here is that both the Target (CommSignal2) and the Interference (EMISignal1) appear to have identical amplitude ranges.
* **Observation:** The "Jammer" is not inherently louder than the Signal in these raw plots.
* **Physical Interpretation:** This is intentional. The dataset specification confirms that all raw frames are **scaled to unity power**.
    * **Traditional Detection:** Often relies on "energy detection" (if energy > threshold, then jammer). This dataset defeats that naive approach immediately.
    * **Deep Learning Logic:** By normalizing power, the problem forces the model to learn the **statistical structure** (the QPSK symbols vs. the chaotic noise) rather than relying on volume.
* **Operational Benefit:** A model trained on this data will be robust against "Low Probability of Intercept" (LPI) jammers that try to hide by matching the power level of the target signal.

**Verdict:** The data preparation forces a "shape-based" rather than "power-based" separation strategy, which is the exact capability needed for the LucidRF Autoencoder.

In [ ]:
if clean_batch is not None and noise_batch is not None:
    # Create grid: Rows = NUM_ROWS, Cols = 2 (Left=Time, Right=Frequency/PSD)
    fig, axes = plt.subplots(NUM_ROWS, 2, figsize=(15, 4 * NUM_ROWS))
    
    if NUM_ROWS == 1: axes = axes.reshape(1, -1)
        
    for i in range(NUM_ROWS):
        row_idx = START_ROW + i
        
        # --- LEFT: Time Domain (What you saw before) ---
        ax_time = axes[i, 0]
        ax_time.plot(clean_batch[i].real, label='Signal (Real)', color='C0', alpha=0.7)
        ax_time.plot(noise_batch[i].real, label='Noise (Real)', color='C1', alpha=0.5)
        ax_time.set_title(f"Row {row_idx}: Time Domain")
        if i==0: ax_time.legend()
        
        # --- RIGHT: Frequency Domain (The Truth) ---
        ax_freq = axes[i, 1]
        
        # Calculate PSD for Clean Signal
        ax_freq.psd(clean_batch[i], NFFT=1024, Fs=1.0, color='C0', label='Target (CommSignal2)')
        
        # Calculate PSD for Noise Signal
        ax_freq.psd(noise_batch[i], NFFT=1024, Fs=1.0, color='C1', label='Interference (EMISignal1)', alpha=0.7)
        
        ax_freq.set_title(f"Row {row_idx}: Frequency Domain (PSD)")
        if i==0: ax_freq.legend()

    plt.tight_layout()
    save_plot(f"mit_dataset_freq_rows_{START_ROW}_to_{START_ROW + NUM_ROWS - 1}.png", machine_id="B", nb_id=NB_ID, fig_id="02")
    plt.show()

# Fig 00.2 The Mixture: The Physics of Co-Channel Interference

The Frequency Domain analysis (Fig 2, Right Column) provides the definitive proof of why standard filters fail and why your Autoencoder is necessary.

### The "Co-Channel" Trap
The Power Spectral Density (PSD) plot reveals the core difficulty of this challenge: **Spectral Overlap**.
* **Observation:** The Interference (Orange) sits directly on top of the Target Signal (Blue).
* **Physical Interpretation:** The interference has been intentionally shifted to the baseband.
    * **Standard Filters:** A Band-Pass Filter works by cutting out specific frequencies (e.g., "ignore everything above 5MHz"). Here, the jammer occupies the *exact same* frequencies as the signal. Cutting the jammer would mean cutting the signal.
    * **The "Blind Spot":** To a conventional receiver, these two signals are mathematically indistinguishable in terms of frequency allocation.

### The Necessity of Non-Linear Separation
Since the signals overlap in both Time and Frequency, they are only separable in the **Feature Space**.
* **The ML Task:** The Autoencoder must find a latent representation where these signals *do not* overlap.
* **Logic:**
    1.  **Decompose:** Break the signal down into abstract features (statistical regularities, modulation patterns).
    2.  **Filter:** Recognize that "Pattern A" (CommSignal2) is valid data, while "Pattern B" (EMISignal1) is noise.
    3.  **Reconstruct:** Rebuild Pattern A while discarding Pattern B.

**Verdict:** The PSD overlap confirms that this is a **Co-Channel Interference** problem. This validates the architectural choice of using an Autoencoder, as linear filters are physically incapable of solving this specific mixture.